# MDN for 3R Inverse Kinematics in 3D

This notebook reproduces the thesis pipeline end-to-end in Google Colab:

1. Define the 3R robot (in 3D) and its forward kinematics (FK).
2. Generate a joint-space-sampled dataset (preserves multiple IK branches), with joint angles represented as **sin/cos pairs** for the network (Section 1.2) rather than raw radians — this avoids the ill-conditioning/branch-confusion failure mode found during error analysis (see Section 14).
3. Train a **Mixture Density Network (MDN)** to predict a mixture distribution over the sin/cos-encoded joint angles given a target position `[x, y, z]`, combining an NLL loss with an FK-consistency loss.
4. Sweep learning rates as a quick diagnostic (Section 10).
5. Evaluate with a best-of-K selection strategy (pick the mixture component whose FK output best matches the target pose).
6. Visualize training curves, workspace coverage, and prediction quality.
7. Check generalization on unseen data.
8. Summarize the most useful diagnostics for improving the model in one place (Section 14) — bias/variance, requirement pass/fail, mixture health, and the theta3 branch-confusion / ill-conditioning check.
9. Package results for download.

**Table of contents:** 1. Robot Parameters & FK &middot; 2. Dataset Generation &middot; 3. Mixture Density Network &middot; 4. Training Utilities &middot; 5. Evaluation: Best-of-K &middot; 6. Experiment I/O & Plotting &middot; 7. Configuration &middot; 8. Build Dataset & Dataloaders &middot; 9. Train the MDN &middot; 10. Learning-Rate Sweep &middot; 11. Evaluate on Test Set &middot; 12. Visualizations &middot; 13. Generalization Check &middot; 14. Key Diagnostics Summary &middot; 15. Save & Download

No GPU is required (the network is a small MLP), but if you want one: **Runtime > Change runtime type > T4 GPU**. The code auto-detects `cuda` if available.

> **Note:** this version of the notebook changed the network's output representation (sin/cos encoding, Section 1.2/3). Any checkpoints saved by an earlier version are architecturally incompatible — run the notebook from the top.

In [ ]:
import math
import json
import shutil
import logging
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Any
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Robot Parameters & Forward Kinematics

### 1.1 Kinematic Model

A 3R robot in 3D:

- `theta1` — base yaw around the z-axis
- `theta2` — shoulder pitch
- `theta3` — elbow pitch

FK model:

```
r = L1 + L2*cos(theta2) + L3*cos(theta2 + theta3)
x = r*cos(theta1)
y = r*sin(theta1)
z = L2*sin(theta2) + L3*sin(theta2 + theta3)
```

This predicts position `[x, y, z]` only (not full 6-DoF pose).

In [ ]:
@dataclass
class RobotParams3R3D:
    """
    3R robot in 3D.

    Kinematic structure:
        theta1 = base yaw around z-axis
        theta2 = shoulder pitch
        theta3 = elbow pitch

    FK model:
        r = L1 + L2*cos(theta2) + L3*cos(theta2 + theta3)

        x = r*cos(theta1)
        y = r*sin(theta1)
        z = L2*sin(theta2) + L3*sin(theta2 + theta3)

    This is a simple 3D position-control robot.
    It predicts x, y, z only, not full 6D pose.
    """

    L1: float = 1.0
    L2: float = 1.0
    L3: float = 0.5

    theta1_lim: Tuple[float, float] = (-math.pi, math.pi)
    theta2_lim: Tuple[float, float] = (-math.pi / 2, math.pi / 2)
    theta3_lim: Tuple[float, float] = (-math.pi / 2, math.pi / 2)

    @property
    def n_dof(self) -> int:
        return 3

    @property
    def input_dim(self) -> int:
        return 3

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def forward_kinematics_numpy(theta: np.ndarray, params: RobotParams3R3D) -> np.ndarray:
    """
    NumPy FK for 3R robot in 3D.

    Args:
        theta: Shape (N, 3), columns theta1, theta2, theta3

    Returns:
        pose: Shape (N, 3), columns x, y, z
    """
    theta = np.asarray(theta, dtype=np.float32)

    t1 = theta[:, 0]
    t2 = theta[:, 1]
    t3 = theta[:, 2]

    r = params.L1 + params.L2 * np.cos(t2) + params.L3 * np.cos(t2 + t3)

    x = r * np.cos(t1)
    y = r * np.sin(t1)
    z = params.L2 * np.sin(t2) + params.L3 * np.sin(t2 + t3)

    return np.stack([x, y, z], axis=1).astype(np.float32)


def forward_kinematics_torch(theta: torch.Tensor, params: RobotParams3R3D) -> torch.Tensor:
    """
    Torch FK for 3R robot in 3D.

    Args:
        theta: Shape (batch, 3) or (batch, components, 3)

    Returns:
        pose: Shape (batch, 1, 3) or (batch, components, 3)
    """
    if theta.dim() == 2:
        theta = theta.unsqueeze(1)

    t1 = theta[:, :, 0]
    t2 = theta[:, :, 1]
    t3 = theta[:, :, 2]

    r = params.L1 + params.L2 * torch.cos(t2) + params.L3 * torch.cos(t2 + t3)

    x = r * torch.cos(t1)
    y = r * torch.sin(t1)
    z = params.L2 * torch.sin(t2) + params.L3 * torch.sin(t2 + t3)

    return torch.stack([x, y, z], dim=2)


def joint_limit_penalty(theta: torch.Tensor, params: RobotParams3R3D) -> torch.Tensor:
    """
    Soft penalty for predictions outside joint limits.

    Args:
        theta: Shape (batch, components, 3)

    Returns:
        penalty: Shape (batch, components)
    """
    limits = torch.tensor(
        [params.theta1_lim, params.theta2_lim, params.theta3_lim],
        dtype=theta.dtype,
        device=theta.device,
    )

    lower = limits[:, 0].view(1, 1, 3)
    upper = limits[:, 1].view(1, 1, 3)

    below = torch.relu(lower - theta)
    above = torch.relu(theta - upper)

    return torch.mean(below**2 + above**2, dim=2)

### 1.2 Angle Encoding for the Network (sin/cos)

The MDN doesn't predict `[theta1, theta2, theta3]` directly — it predicts `[sin1, cos1, sin2, cos2, sin3, cos3]`.

Two reasons this matters for this specific model:

1. **`theta1` wraps around the full circle** (`-pi` to `pi`). A raw-radian regression target treats `-pi` and `pi` as maximally far apart even though they're the same angle — sin/cos encoding removes that discontinuity entirely.
2. **FK itself is expressed in sin/cos of the angles**, not the angles themselves (`cos(theta2)`, `cos(theta2+theta3)`, ...). Making the network predict sin/cos directly removes a layer of nonlinearity it would otherwise have to learn, and — more importantly for this project — it directly targets the failure mode found during error analysis: `theta2`/`theta3` MAE was far higher than `theta1`'s despite a tiny FK/pose error, the signature of the model occasionally landing on the *other* valid elbow-up/elbow-down branch (which is a sign flip in `theta3`) or of the elbow-angle solution's inherent ill-conditioning near full extension/fold (`theta3 -> 0`). Section 14 quantifies both effects directly on this representation.

`decode_sincos_to_theta_torch` recovers the angle via `atan2(sin, cos)` wherever a real angle is needed (FK, joint limits, reporting) — this is exact (no information lost by the encoding) and correctly handles the full `-pi..pi` range.

In [ ]:
ANGLE_ENCODING_DIM = 6  # [sin1, cos1, sin2, cos2, sin3, cos3] — replaces raw [theta1, theta2, theta3] as the network's target/output space


def encode_theta_to_sincos(theta: np.ndarray) -> np.ndarray:
    """
    Encode raw joint angles as sin/cos pairs.

    Args:
        theta: Shape (..., 3) [theta1, theta2, theta3], radians

    Returns:
        sincos: Shape (..., 6) [sin1, cos1, sin2, cos2, sin3, cos3]
    """
    theta = np.asarray(theta, dtype=np.float32)

    sin = np.sin(theta)
    cos = np.cos(theta)

    return np.stack(
        [sin[..., 0], cos[..., 0], sin[..., 1], cos[..., 1], sin[..., 2], cos[..., 2]],
        axis=-1,
    ).astype(np.float32)


def normalize_sincos_torch(sincos: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Rescale each (sin, cos) pair in a (..., 6) tensor back onto the unit circle."""
    pairs = sincos.view(*sincos.shape[:-1], 3, 2)
    norm = torch.norm(pairs, dim=-1, keepdim=True).clamp_min(eps)
    return (pairs / norm).view(*sincos.shape[:-1], 6)


def decode_sincos_to_theta_torch(sincos: torch.Tensor) -> torch.Tensor:
    """
    Decode sin/cos-encoded angles back to radians via atan2.

    Args:
        sincos: Shape (..., 6) [sin1, cos1, sin2, cos2, sin3, cos3]

    Returns:
        theta: Shape (..., 3) [theta1, theta2, theta3], radians
    """
    pairs = sincos.view(*sincos.shape[:-1], 3, 2)
    return torch.atan2(pairs[..., 0], pairs[..., 1])


def unit_circle_penalty(sincos: torch.Tensor) -> torch.Tensor:
    """
    Soft penalty keeping each predicted (sin, cos) pair near the unit circle
    (a diagonal Gaussian over 6 dims doesn't know sin^2+cos^2=1 on its own).

    Args:
        sincos: Shape (..., 6)

    Returns:
        penalty: Shape (...,) — one value per leading-dim entry (e.g. per batch*component)
    """
    pairs = sincos.view(*sincos.shape[:-1], 3, 2)
    radius_sq = torch.sum(pairs**2, dim=-1)
    return torch.mean((radius_sq - 1.0) ** 2, dim=-1)

## 2. Dataset Generation

We sample **joint space** uniformly (not Cartesian space) and run FK to get positions. This preserves multiple valid IK branches per position, which is exactly what the MDN needs to learn a multi-modal distribution.

In [ ]:
def generate_dataset(n_samples: int, params: RobotParams3R3D, seed: int = 42):
    """
    Generate dataset using joint-space sampling.

    Returns:
        features: Shape (N, 3) [x, y, z]
        targets:  Shape (N, 3) [theta1, theta2, theta3]
    """
    rng = np.random.default_rng(seed)

    theta1 = rng.uniform(params.theta1_lim[0], params.theta1_lim[1], size=n_samples)
    theta2 = rng.uniform(params.theta2_lim[0], params.theta2_lim[1], size=n_samples)
    theta3 = rng.uniform(params.theta3_lim[0], params.theta3_lim[1], size=n_samples)

    targets = np.stack([theta1, theta2, theta3], axis=1).astype(np.float32)
    features = forward_kinematics_numpy(targets, params).astype(np.float32)

    return features, targets


def split_and_normalize(
    features,
    targets,
    train_fraction=0.8,
    val_fraction=0.1,
    test_fraction=0.1,
    seed=0,
):
    """
    Split and normalize dataset. Returns raw data, normalized data, indices, and stats.

    Targets are stored two ways:
        - `targets_original`: raw [theta1, theta2, theta3] radians — used for metrics/reporting.
        - `targets_normalized`: sin/cos-encoded [sin1,cos1,sin2,cos2,sin3,cos3] — the actual
          training target (see Section 1.2). Sin/cos values are already bounded to [-1, 1],
          so no z-score standardization is applied to them; `theta_mean`/`theta_std` are kept
          as identity (zeros/ones) purely so downstream code that broadcasts against them
          (`mu_norm * theta_std + theta_mean`) keeps working unchanged.
    """
    assert abs(train_fraction + val_fraction + test_fraction - 1.0) < 1e-6

    n_samples = len(features)

    rng = np.random.default_rng(seed)
    indices = np.arange(n_samples)
    rng.shuffle(indices)

    n_train = int(train_fraction * n_samples)
    n_val = int(val_fraction * n_samples)

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]
    test_idx = indices[n_train + n_val:]

    feature_mean = features[train_idx].mean(axis=0, keepdims=True)
    feature_std = features[train_idx].std(axis=0, keepdims=True) + 1e-8

    targets_sincos = encode_theta_to_sincos(targets)

    theta_mean = np.zeros((1, ANGLE_ENCODING_DIM), dtype=np.float32)
    theta_std = np.ones((1, ANGLE_ENCODING_DIM), dtype=np.float32)

    features_norm = (features - feature_mean) / feature_std
    targets_norm = (targets_sincos - theta_mean) / theta_std

    return {
        "features_original": features.astype(np.float32),
        "targets_original": targets.astype(np.float32),
        "features_normalized": features_norm.astype(np.float32),
        "targets_normalized": targets_norm.astype(np.float32),
        "train_indices": train_idx,
        "val_indices": val_idx,
        "test_indices": test_idx,
        "feature_mean": feature_mean.astype(np.float32),
        "feature_std": feature_std.astype(np.float32),
        "theta_mean": theta_mean.astype(np.float32),
        "theta_std": theta_std.astype(np.float32),
    }

## 3. Mixture Density Network

Input: `[x, y, z]`. Output: a mixture of `K` Gaussians over the sin/cos-encoded joint angles — mixture weights `pi`, means `mu`, and standard deviations `sigma`. For every pose, the model proposes `K` candidate joint configurations.

In [ ]:
class MDN3R3D(nn.Module):
    """
    Mixture Density Network for 3R inverse kinematics in 3D.

    Input:
        [x, y, z]

    Output:
        Mixture distribution over the sin/cos-encoded joint angles:
            [sin1, cos1, sin2, cos2, sin3, cos3]  (see Section 1.2)

    For every pose, the model predicts K possible joint configurations.
    """

    def __init__(
        self,
        input_dim: int = 3,
        output_dim: int = ANGLE_ENCODING_DIM,
        hidden_size: int = 256,
        n_components: int = 8,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.n_components = n_components

        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
        )

        final_dim = hidden_size // 2

        self.pi_head = nn.Linear(final_dim, n_components)
        self.mu_head = nn.Linear(final_dim, n_components * output_dim)
        self.log_sigma_head = nn.Linear(final_dim, n_components * output_dim)

    def forward(self, x):
        """
        Args:
            x: Shape (batch, input_dim)

        Returns:
            pi:    Shape (batch, components)
            mu:    Shape (batch, components, output_dim)
            sigma: Shape (batch, components, output_dim)
        """
        h = self.backbone(x)

        pi = torch.softmax(self.pi_head(h), dim=1)

        mu = self.mu_head(h).view(-1, self.n_components, self.output_dim)

        log_sigma = self.log_sigma_head(h).view(-1, self.n_components, self.output_dim)
        log_sigma = torch.clamp(log_sigma, min=-5.0, max=2.0)

        sigma = torch.exp(log_sigma)

        return pi, mu, sigma

    def mdn_loss(self, pi, mu, sigma, target):
        """
        Negative log likelihood loss for MDN.

        target is the (unnormalized) sin/cos-encoded joint angle: Shape (batch, output_dim)
        """
        target = target.unsqueeze(1)

        var = sigma**2

        log_prob = -0.5 * torch.sum(((target - mu) ** 2) / var, dim=2)
        log_prob += -0.5 * self.output_dim * math.log(2.0 * math.pi)
        log_prob += -torch.sum(torch.log(sigma + 1e-8), dim=2)

        weighted_log_prob = log_prob + torch.log(pi + 1e-8)

        log_likelihood = torch.logsumexp(weighted_log_prob, dim=1)

        return -torch.mean(log_likelihood)

    def predict_best_fk(
        self,
        x_norm,
        x_original,
        theta_mean,
        theta_std,
        params: RobotParams3R3D,
        probability_weight: float = 0.01,
        limit_weight: float = 0.05,
        circle_weight: float = 0.05,
    ):
        """
        Predict best joint configuration using FK consistency.

        The model predicts multiple candidate joint solutions (as sin/cos pairs). We
        decode each to radians, run FK on every candidate, and select the one with the
        lowest physical pose error (penalized for joint-limit violations and for
        straying off the unit circle).
        """
        self.eval()

        with torch.no_grad():
            pi, mu_norm, sigma = self.forward(x_norm)

            mu_sincos = mu_norm * theta_std + theta_mean
            mu_theta = decode_sincos_to_theta_torch(mu_sincos)

            fk_candidates = forward_kinematics_torch(mu_theta, params)

            target_pose = x_original.unsqueeze(1)

            fk_error = torch.mean((fk_candidates - target_pose) ** 2, dim=2)

            limit_penalty = joint_limit_penalty(mu_theta, params)
            circle_penalty = unit_circle_penalty(mu_sincos)

            score = (
                fk_error
                + limit_weight * limit_penalty
                + circle_weight * circle_penalty
                - probability_weight * torch.log(pi + 1e-8)
            )

            best_component = torch.argmin(score, dim=1)

            batch_indices = torch.arange(x_norm.shape[0], device=x_norm.device)

            best_theta = mu_theta[batch_indices, best_component]

            info = {
                "pi": pi,
                "mu_theta": mu_theta,
                "fk_candidates": fk_candidates,
                "fk_error": fk_error,
                "best_component": best_component,
            }

            return best_theta, info

## 4. Training Utilities

- `IKDataset3D` wraps normalized inputs/targets plus the original (unnormalized) pose, needed for the FK-consistency loss.
- `fk_consistency_loss` runs FK on every mixture component's mean and keeps the **best** (lowest-error) candidate, so the network isn't punished for other components disagreeing.
- `train_mdn` combines NLL + `lambda_fk * FK loss`, with gradient clipping, early stopping, and best-checkpoint saving.

In [ ]:
class IKDataset3D(Dataset):
    """Dataset returns: x_norm, x_original, y_norm"""

    def __init__(self, x_norm, x_original, y_norm):
        self.x_norm = torch.tensor(x_norm, dtype=torch.float32)
        self.x_original = torch.tensor(x_original, dtype=torch.float32)
        self.y_norm = torch.tensor(y_norm, dtype=torch.float32)

    def __len__(self):
        return len(self.x_norm)

    def __getitem__(self, idx):
        return self.x_norm[idx], self.x_original[idx], self.y_norm[idx]


def fk_consistency_loss(
    mu_sincos, x_original, params: RobotParams3R3D,
    limit_weight: float = 0.05, circle_weight: float = 0.05,
):
    """
    FK consistency loss. Uses the best FK candidate among all mixture components.

    `mu_sincos` is the mixture means in sin/cos-encoded space (Shape (batch, components, 6));
    it's decoded to radians before FK/joint-limit checks, and also penalized directly for
    straying off the unit circle (a diagonal Gaussian over sin/cos doesn't enforce sin^2+cos^2=1).
    """
    mu_theta = decode_sincos_to_theta_torch(mu_sincos)

    fk_candidates = forward_kinematics_torch(mu_theta, params)

    target_pose = x_original.unsqueeze(1)

    fk_error = torch.mean((fk_candidates - target_pose) ** 2, dim=2)

    limit_penalty = joint_limit_penalty(mu_theta, params)
    circle_penalty = unit_circle_penalty(mu_sincos)

    total_error = fk_error + limit_weight * limit_penalty + circle_weight * circle_penalty

    best_error = torch.min(total_error, dim=1)[0]

    return best_error.mean()


def evaluate_loss(model, loader, device, theta_mean_t, theta_std_t, params, lambda_fk):
    model.eval()

    total_loss = 0.0
    total_nll = 0.0
    total_fk = 0.0
    total_count = 0

    with torch.no_grad():
        for x_norm, x_original, y_norm in loader:
            x_norm = x_norm.to(device)
            x_original = x_original.to(device)
            y_norm = y_norm.to(device)

            pi, mu_norm, sigma = model(x_norm)

            nll_loss = model.mdn_loss(pi, mu_norm, sigma, y_norm)

            mu_real = mu_norm * theta_std_t + theta_mean_t

            fk_loss = fk_consistency_loss(mu_real, x_original, params)

            loss = nll_loss + lambda_fk * fk_loss

            batch_size = x_norm.shape[0]

            total_loss += loss.item() * batch_size
            total_nll += nll_loss.item() * batch_size
            total_fk += fk_loss.item() * batch_size
            total_count += batch_size

    return {
        "loss": total_loss / total_count,
        "nll_loss": total_nll / total_count,
        "fk_loss": total_fk / total_count,
    }


def save_checkpoint(
    path, model, optimizer, epoch, best_val_loss,
    feature_mean, feature_std, theta_mean, theta_std, params, config,
):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "theta_mean": theta_mean,
        "theta_std": theta_std,
        "robot_params": params.to_dict(),
        "training_config": config,
    }
    torch.save(checkpoint, path)


def train_mdn(
    model, train_loader, val_loader, optimizer, device, params,
    theta_mean, theta_std, feature_mean, feature_std,
    config, experiment_dirs, logger,
):
    """Train MDN with: MDN NLL loss, FK consistency loss, early stopping, best checkpoint saving."""
    max_epochs = config["max_epochs"]
    patience = config["patience"]
    min_delta = config["min_delta"]
    lambda_fk = config["lambda_fk"]

    model.to(device)

    theta_mean_t = torch.tensor(theta_mean, dtype=torch.float32, device=device).view(1, 1, -1)
    theta_std_t = torch.tensor(theta_std, dtype=torch.float32, device=device).view(1, 1, -1)

    best_val_loss = float("inf")
    best_epoch = 0
    no_improvement_count = 0
    early_stopped = False

    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()

        total_loss = 0.0
        total_nll = 0.0
        total_fk = 0.0
        total_count = 0

        for x_norm, x_original, y_norm in train_loader:
            x_norm = x_norm.to(device)
            x_original = x_original.to(device)
            y_norm = y_norm.to(device)

            optimizer.zero_grad()

            pi, mu_norm, sigma = model(x_norm)

            nll_loss = model.mdn_loss(pi, mu_norm, sigma, y_norm)

            mu_real = mu_norm * theta_std_t + theta_mean_t

            fk_loss = fk_consistency_loss(mu_real, x_original, params)

            loss = nll_loss + lambda_fk * fk_loss

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()

            batch_size = x_norm.shape[0]

            total_loss += loss.item() * batch_size
            total_nll += nll_loss.item() * batch_size
            total_fk += fk_loss.item() * batch_size
            total_count += batch_size

        train_loss = total_loss / total_count
        train_nll = total_nll / total_count
        train_fk = total_fk / total_count

        val_stats = evaluate_loss(
            model=model, loader=val_loader, device=device,
            theta_mean_t=theta_mean_t, theta_std_t=theta_std_t,
            params=params, lambda_fk=lambda_fk,
        )

        val_loss = val_stats["loss"]
        val_nll = val_stats["nll_loss"]
        val_fk = val_stats["fk_loss"]

        learning_rate = optimizer.param_groups[0]["lr"]

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_nll_loss": train_nll,
            "train_fk_loss": train_fk,
            "val_nll_loss": val_nll,
            "val_fk_loss": val_fk,
            "learning_rate": learning_rate,
        }

        history.append(row)

        if epoch == 1 or epoch % 10 == 0:
            logger.info(
                f"Epoch {epoch}/{max_epochs}, Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}"
            )

        improved = val_loss < best_val_loss - min_delta

        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            no_improvement_count = 0

            best_model_path = experiment_dirs["models"] / "best_model.pt"
            best_checkpoint_path = experiment_dirs["models"] / "checkpoint_best.pt"

            torch.save(model.state_dict(), best_model_path)

            save_checkpoint(
                path=best_checkpoint_path, model=model, optimizer=optimizer,
                epoch=epoch, best_val_loss=best_val_loss,
                feature_mean=feature_mean, feature_std=feature_std,
                theta_mean=theta_mean, theta_std=theta_std,
                params=params, config=config,
            )

            logger.info(f"Validation improved to {best_val_loss:.6f} at epoch {epoch}.")
        else:
            no_improvement_count += 1

        if no_improvement_count >= patience:
            early_stopped = True
            logger.info(f"Early stopping triggered after {patience} epochs without improvement.")
            break

    final_model_path = experiment_dirs["models"] / "final_model.pt"
    torch.save(model.state_dict(), final_model_path)
    logger.info(f"Saved final model to {final_model_path}.")

    return {
        "history": history,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "total_epochs_trained": len(history),
        "early_stopped": early_stopped,
    }

## 5. Evaluation: Best-of-K

For each input pose, the MDN proposes `K` candidate joint configurations. We score every candidate by running FK and picking the one whose predicted pose is closest to the target (tie-broken by mixture probability), then report angle and pose errors for that best candidate.

In [ ]:
def evaluate_best_of_k(model, loader, device, params, theta_mean, theta_std):
    """
    Best-of-K evaluation.

    For each input, the MDN predicts K candidate joint configurations (as sin/cos pairs,
    decoded to radians via atan2). We compute the best joint-angle error, best FK pose
    error, and best component index, in radian/position space — same metric definitions
    as before the sin/cos encoding change.
    """
    model.eval()

    theta_mean_t = torch.tensor(theta_mean, dtype=torch.float32, device=device).view(1, 1, -1)
    theta_std_t = torch.tensor(theta_std, dtype=torch.float32, device=device).view(1, 1, -1)

    all_x_original = []
    all_y_true_real = []
    all_mu_real = []
    all_pi = []
    all_fk_candidates = []
    all_best_component = []
    all_best_fk_pose_mse = []
    all_best_angle_mse = []
    all_best_pred = []

    with torch.no_grad():
        for x_norm, x_original, y_norm in loader:
            x_norm = x_norm.to(device)
            x_original = x_original.to(device)
            y_norm = y_norm.to(device)

            pi, mu_norm, sigma = model(x_norm)

            mu_sincos = mu_norm * theta_std_t + theta_mean_t
            mu_theta = decode_sincos_to_theta_torch(mu_sincos)

            y_sincos = (
                y_norm * theta_std_t.squeeze(0).squeeze(0)
                + theta_mean_t.squeeze(0).squeeze(0)
            )
            y_true_real = decode_sincos_to_theta_torch(y_sincos)

            fk_candidates = forward_kinematics_torch(mu_theta, params)

            pose_target = x_original.unsqueeze(1)

            fk_error = torch.mean((fk_candidates - pose_target) ** 2, dim=2)

            y_expanded = y_true_real.unsqueeze(1)

            angle_error = torch.mean((mu_theta - y_expanded) ** 2, dim=2)

            score = fk_error - 0.01 * torch.log(pi + 1e-8)

            best_component = torch.argmin(score, dim=1)

            batch_indices = torch.arange(x_norm.shape[0], device=device)

            best_pred = mu_theta[batch_indices, best_component]

            best_fk_pose_mse = fk_error[batch_indices, best_component]
            best_angle_mse = angle_error[batch_indices, best_component]

            all_x_original.append(x_original.cpu().numpy())
            all_y_true_real.append(y_true_real.cpu().numpy())
            all_mu_real.append(mu_theta.cpu().numpy())
            all_pi.append(pi.cpu().numpy())
            all_fk_candidates.append(fk_candidates.cpu().numpy())
            all_best_component.append(best_component.cpu().numpy())
            all_best_fk_pose_mse.append(best_fk_pose_mse.cpu().numpy())
            all_best_angle_mse.append(best_angle_mse.cpu().numpy())
            all_best_pred.append(best_pred.cpu().numpy())

    x_original = np.concatenate(all_x_original, axis=0)
    y_true_real = np.concatenate(all_y_true_real, axis=0)
    predicted_all_components_real = np.concatenate(all_mu_real, axis=0)
    mixture_probabilities = np.concatenate(all_pi, axis=0)
    fk_pose_candidates = np.concatenate(all_fk_candidates, axis=0)
    best_component_indices = np.concatenate(all_best_component, axis=0)
    best_fk_pose_mse = np.concatenate(all_best_fk_pose_mse, axis=0)
    best_angle_mse = np.concatenate(all_best_angle_mse, axis=0)
    predicted_best_fk_real = np.concatenate(all_best_pred, axis=0)

    metrics = {
        "best_of_k_angle_mse": float(np.mean(best_angle_mse)),
        "best_of_k_angle_rmse": float(math.sqrt(np.mean(best_angle_mse))),
        "best_of_k_pose_mse": float(np.mean(best_fk_pose_mse)),
        "best_of_k_pose_rmse": float(math.sqrt(np.mean(best_fk_pose_mse))),
        "theta1_mae": float(np.mean(np.abs(predicted_best_fk_real[:, 0] - y_true_real[:, 0]))),
        "theta2_mae": float(np.mean(np.abs(predicted_best_fk_real[:, 1] - y_true_real[:, 1]))),
        "theta3_mae": float(np.mean(np.abs(predicted_best_fk_real[:, 2] - y_true_real[:, 2]))),
    }

    predictions = {
        "x_original": x_original,
        "y_true_real": y_true_real,
        "predicted_all_components_real": predicted_all_components_real,
        "predicted_best_fk_real": predicted_best_fk_real,
        "mixture_probabilities": mixture_probabilities,
        "fk_pose_candidates": fk_pose_candidates,
        "best_component_indices": best_component_indices,
        "best_fk_pose_mse": best_fk_pose_mse,
        "best_angle_mse": best_angle_mse,
    }

    return metrics, predictions

## 6. Experiment I/O & Plotting Utilities

Sets up an `outputs/experiments/<timestamp>/` directory (mirrors the local thesis pipeline layout) and helpers to save JSON/NPZ/CSV plus the standard diagnostic plots.

In [ ]:
def create_experiment_dirs(base_dir="outputs/experiments", run_name=None):
    if run_name is None:
        run_name = datetime.now().strftime("%Y%m%d_%H%M%S")

    root = Path(base_dir) / run_name

    dirs = {
        "root": root,
        "models": root / "models",
        "dataset": root / "dataset",
        "processed_data": root / "processed_data",
        "metrics": root / "metrics",
        "plots": root / "plots",
        "configs": root / "configs",
        "logs": root / "logs",
        "predictions": root / "predictions",
        "unseen_root": root / "unseen_data",
        "unseen_dataset": root / "unseen_data" / "dataset",
        "unseen_predictions": root / "unseen_data" / "predictions",
        "unseen_metrics": root / "unseen_data" / "metrics",
        "unseen_tables": root / "unseen_data" / "tables",
        "unseen_plots": root / "unseen_data" / "plots",
    }

    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)

    return dirs


def setup_logger(log_path: Path):
    logger = logging.getLogger("ik_mdn_3d")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    file_handler = logging.FileHandler(log_path)
    file_handler.setFormatter(formatter)

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    return logger


def save_json(data, path):
    path = Path(path)

    def convert(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.float32, np.float64)):
            return float(obj)
        if isinstance(obj, (np.int32, np.int64)):
            return int(obj)
        return obj

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, default=convert)


def save_npz(data_dict, path):
    np.savez_compressed(path, **data_dict)


def save_train_history(history, metrics_dir):
    df = pd.DataFrame(history)
    df.to_json(metrics_dir / "train_history.json", orient="records", indent=4)
    df.to_csv(metrics_dir / "train_history.csv", index=False)


def save_final_metrics(metrics, metrics_dir):
    save_json(metrics, metrics_dir / "final_metrics.json")
    pd.DataFrame([metrics]).to_csv(metrics_dir / "final_metrics.csv", index=False)


def prediction_summary_dataframe(predictions):
    x = predictions["x_original"]
    y_true = predictions["y_true_real"]
    y_pred = predictions["predicted_best_fk_real"]
    pi = predictions["mixture_probabilities"]
    best_component = predictions["best_component_indices"]

    rows = []
    for i in range(len(x)):
        selected = int(best_component[i])
        rows.append({
            "sample_id": i,
            "x": x[i, 0], "y": x[i, 1], "z": x[i, 2],
            "theta1_true": y_true[i, 0], "theta2_true": y_true[i, 1], "theta3_true": y_true[i, 2],
            "theta1_pred": y_pred[i, 0], "theta2_pred": y_pred[i, 1], "theta3_pred": y_pred[i, 2],
            "best_component": selected,
            "best_fk_pose_mse": predictions["best_fk_pose_mse"][i],
            "best_angle_mse": predictions["best_angle_mse"][i],
            "max_mixture_probability": np.max(pi[i]),
            "selected_component_probability": pi[i, selected],
        })

    return pd.DataFrame(rows)


def component_usage_dataframe(predictions, n_components):
    best_component = predictions["best_component_indices"]
    counts = np.bincount(best_component, minlength=n_components)
    total = len(best_component)

    rows = [
        {"component_id": cid, "selected_count": int(count), "selected_percentage": float(100.0 * count / total)}
        for cid, count in enumerate(counts)
    ]
    return pd.DataFrame(rows)


def error_summary_dataframe(predictions):
    y_true = predictions["y_true_real"]
    y_pred = predictions["predicted_best_fk_real"]

    values = {
        "best_fk_pose_mse": predictions["best_fk_pose_mse"],
        "best_angle_mse": predictions["best_angle_mse"],
        "theta1_abs_error": np.abs(y_pred[:, 0] - y_true[:, 0]),
        "theta2_abs_error": np.abs(y_pred[:, 1] - y_true[:, 1]),
        "theta3_abs_error": np.abs(y_pred[:, 2] - y_true[:, 2]),
    }

    rows = [
        {"metric_name": name, "mean": float(np.mean(arr)), "std": float(np.std(arr)),
         "min": float(np.min(arr)), "median": float(np.median(arr)), "max": float(np.max(arr))}
        for name, arr in values.items()
    ]
    return pd.DataFrame(rows)


def save_prediction_tables(predictions, output_dir, n_components, prefix="test"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    prediction_summary_dataframe(predictions).to_csv(output_dir / f"{prefix}_prediction_summary.csv", index=False)
    component_usage_dataframe(predictions, n_components).to_csv(output_dir / f"{prefix}_component_usage.csv", index=False)
    error_summary_dataframe(predictions).to_csv(output_dir / f"{prefix}_error_summary.csv", index=False)


def save_training_plots(history, plots_dir):
    df = pd.DataFrame(history)

    plt.figure(figsize=(8, 5))
    plt.plot(df["epoch"], df["train_loss"], label="train_loss")
    plt.plot(df["epoch"], df["val_loss"], label="val_loss")

    best_idx = df["val_loss"].idxmin()
    best_epoch = df.loc[best_idx, "epoch"]
    best_val = df.loc[best_idx, "val_loss"]

    plt.scatter([best_epoch], [best_val], marker="o", label="best_val_epoch")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training and Validation Loss")
    plt.legend(); plt.tight_layout()
    plt.savefig(plots_dir / "training_loss_curve.png", dpi=150)
    plt.show()
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(df["epoch"], df["train_nll_loss"], label="train_nll")
    plt.plot(df["epoch"], df["val_nll_loss"], label="val_nll")
    plt.plot(df["epoch"], df["train_fk_loss"], label="train_fk")
    plt.plot(df["epoch"], df["val_fk_loss"], label="val_fk")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("NLL and FK Loss")
    plt.legend(); plt.tight_layout()
    plt.savefig(plots_dir / "nll_fk_loss_curve.png", dpi=150)
    plt.show()
    plt.close()


def save_dataset_plots(features_original, plots_dir):
    x = features_original[:, 0]
    y = features_original[:, 1]
    z = features_original[:, 2]

    plt.figure(figsize=(6, 5))
    plt.scatter(x, y, s=2, alpha=0.35)
    plt.xlabel("x"); plt.ylabel("y"); plt.title("Workspace Distribution: x-y")
    plt.axis("equal"); plt.tight_layout()
    plt.savefig(plots_dir / "workspace_xy_distribution.png", dpi=150)
    plt.show()
    plt.close()

    plt.figure(figsize=(6, 5))
    plt.scatter(x, z, s=2, alpha=0.35)
    plt.xlabel("x"); plt.ylabel("z"); plt.title("Workspace Distribution: x-z")
    plt.tight_layout()
    plt.savefig(plots_dir / "workspace_xz_distribution.png", dpi=150)
    plt.show()
    plt.close()


def save_prediction_plots(predictions, plots_dir, n_components, prefix="test"):
    y_true = predictions["y_true_real"]
    y_pred = predictions["predicted_best_fk_real"]
    best_fk = predictions["best_fk_pose_mse"]
    best_angle = predictions["best_angle_mse"]
    best_component = predictions["best_component_indices"]
    x_original = predictions["x_original"]

    plt.figure(figsize=(7, 5))
    plt.hist(best_fk, bins=50)
    plt.xlabel("Best-FK pose MSE"); plt.ylabel("Count"); plt.title(f"{prefix}: FK Pose Error Histogram")
    plt.tight_layout()
    plt.savefig(plots_dir / f"{prefix}_fk_error_histogram.png", dpi=150)
    plt.show()
    plt.close()

    plt.figure(figsize=(7, 5))
    plt.hist(best_angle, bins=50)
    plt.xlabel("Best-of-K angle MSE"); plt.ylabel("Count"); plt.title(f"{prefix}: Angle Error Histogram")
    plt.tight_layout()
    plt.savefig(plots_dir / f"{prefix}_angle_error_histogram.png", dpi=150)
    plt.show()
    plt.close()

    sample_count = min(1000, len(y_true))
    idx = np.arange(sample_count)

    fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
    names = ["theta1", "theta2", "theta3"]

    for j, ax in enumerate(axes):
        ax.plot(idx, y_true[:sample_count, j], label="true", linewidth=1)
        ax.plot(idx, y_pred[:sample_count, j], label="pred", linewidth=1)
        ax.set_ylabel(names[j]); ax.legend()

    axes[-1].set_xlabel("Sample index")
    fig.suptitle(f"{prefix}: Predicted vs True Joint Angles")
    fig.tight_layout()
    fig.savefig(plots_dir / f"{prefix}_predicted_vs_true_angles.png", dpi=150)
    plt.show()
    plt.close(fig)

    counts = np.bincount(best_component, minlength=n_components)

    plt.figure(figsize=(7, 5))
    plt.bar(np.arange(n_components), counts)
    plt.xlabel("MDN component"); plt.ylabel("Selected count"); plt.title(f"{prefix}: Mixture Component Usage")
    plt.tight_layout()
    plt.savefig(plots_dir / f"{prefix}_component_usage.png", dpi=150)
    plt.show()
    plt.close()

    plt.figure(figsize=(6, 5))
    plt.scatter(x_original[:, 0], x_original[:, 1], c=best_fk, s=5, alpha=0.7)
    plt.xlabel("x"); plt.ylabel("y"); plt.title(f"{prefix}: Workspace Colored by FK Error")
    plt.colorbar(label="FK pose MSE")
    plt.axis("equal"); plt.tight_layout()
    plt.savefig(plots_dir / f"{prefix}_workspace_error_xy.png", dpi=150)
    plt.show()
    plt.close()

## 7. Configuration & Experiment Setup

Same hyperparameters as the local thesis pipeline (`config.json`).

In [ ]:
config = {
    "project_name": "ik_mdn_3r_3d",
    "n_samples": 15000,
    "batch_size": 256,
    "max_epochs": 200,
    "patience": 20,
    "min_delta": 1e-5,
    "learning_rate": 5e-4,
    "lambda_fk": 0.1,
    "n_components": 8,
    "hidden_size": 256,
    "train_fraction": 0.8,
    "val_fraction": 0.1,
    "test_fraction": 0.1,
    "dataset_seed": 42,
    "split_seed": 0,
    "unseen_seed": 999,
}

experiment_dirs = create_experiment_dirs()
logger = setup_logger(experiment_dirs["logs"] / "training.log")

logger.info(f"Experiment directory: {experiment_dirs['root']}")
logger.info(f"Using device: {device}")

params = RobotParams3R3D()
config["robot_params"] = params.to_dict()

save_json(config, experiment_dirs["configs"] / "config.json")
logger.info("Saved config.")

## 8. Build Dataset & Dataloaders

In [ ]:
def make_loader(processed, indices, batch_size, shuffle=False):
    return DataLoader(
        IKDataset3D(
            processed["features_normalized"][indices],
            processed["features_original"][indices],
            processed["targets_normalized"][indices],
        ),
        batch_size=batch_size,
        shuffle=shuffle,
    )


features, targets = generate_dataset(
    n_samples=config["n_samples"], params=params, seed=config["dataset_seed"],
)

save_npz({"features": features, "targets": targets}, experiment_dirs["dataset"] / "raw_dataset.npz")
logger.info("Saved raw dataset.")

processed = split_and_normalize(
    features=features, targets=targets,
    train_fraction=config["train_fraction"],
    val_fraction=config["val_fraction"],
    test_fraction=config["test_fraction"],
    seed=config["split_seed"],
)

save_npz(processed, experiment_dirs["processed_data"] / "processed_dataset.npz")
logger.info("Saved processed dataset.")

train_loader = make_loader(processed, processed["train_indices"], batch_size=config["batch_size"], shuffle=True)
val_loader = make_loader(processed, processed["val_indices"], batch_size=config["batch_size"], shuffle=False)
test_loader = make_loader(processed, processed["test_indices"], batch_size=config["batch_size"], shuffle=False)

print(f"Train: {len(processed['train_indices'])}  Val: {len(processed['val_indices'])}  Test: {len(processed['test_indices'])}")

## 9. Train the MDN

This runs the full training loop (NLL + FK-consistency loss, early stopping, best-checkpoint saving). With the default config (15k samples, up to 200 epochs, patience 20) this typically takes a few minutes on CPU and under a minute on a Colab GPU.

In [ ]:
model = MDN3R3D(
    input_dim=params.input_dim,
    output_dim=ANGLE_ENCODING_DIM,
    hidden_size=config["hidden_size"],
    n_components=config["n_components"],
)

optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])

train_result = train_mdn(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    params=params,
    theta_mean=processed["theta_mean"],
    theta_std=processed["theta_std"],
    feature_mean=processed["feature_mean"],
    feature_std=processed["feature_std"],
    config=config,
    experiment_dirs=experiment_dirs,
    logger=logger,
)

save_train_history(train_result["history"], experiment_dirs["metrics"])
logger.info("Saved training history JSON and CSV.")

print(f"Best epoch: {train_result['best_epoch']}, Best val loss: {train_result['best_val_loss']:.6f}")
print(f"Early stopped: {train_result['early_stopped']}, Total epochs trained: {train_result['total_epochs_trained']}")

## 10. Learning-Rate Sweep (Diagnostic)

Trains a fresh model from scratch at each of 5 learning rates, entirely in memory (no checkpoints, no log file) — a quick check of how sensitive this setup is to the LR choice before committing to a full run. Takes ~5x as long as one normal training run (reuses `max_epochs`/`patience` from `config` for every LR).

In [ ]:
# Learning-Rate Sweep (diagnostic only) — trains a fresh model from scratch for each
# learning rate below, entirely in memory. Nothing is written to disk; only metrics
# are printed / returned as a DataFrame.

learning_rates_to_try = [1e-2, 1e-3, 5e-4, 1e-4, 1e-5]


def train_mdn_in_memory(
    model, train_loader, val_loader, optimizer, device, params,
    theta_mean, theta_std, max_epochs, patience, min_delta, lambda_fk,
):
    """Same training logic as train_mdn, but keeps everything in memory: no checkpoints, no log file."""
    model.to(device)

    theta_mean_t = torch.tensor(theta_mean, dtype=torch.float32, device=device).view(1, 1, -1)
    theta_std_t = torch.tensor(theta_std, dtype=torch.float32, device=device).view(1, 1, -1)

    best_val_loss = float("inf")
    best_epoch = 0
    no_improvement_count = 0
    early_stopped = False
    final_train_loss = None
    final_val_loss = None

    for epoch in range(1, max_epochs + 1):
        model.train()

        total_loss = 0.0
        total_count = 0

        for x_norm, x_original, y_norm in train_loader:
            x_norm = x_norm.to(device)
            x_original = x_original.to(device)
            y_norm = y_norm.to(device)

            optimizer.zero_grad()

            pi, mu_norm, sigma = model(x_norm)
            nll_loss = model.mdn_loss(pi, mu_norm, sigma, y_norm)

            mu_real = mu_norm * theta_std_t + theta_mean_t
            fk_loss = fk_consistency_loss(mu_real, x_original, params)

            loss = nll_loss + lambda_fk * fk_loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            batch_size = x_norm.shape[0]
            total_loss += loss.item() * batch_size
            total_count += batch_size

        final_train_loss = total_loss / total_count

        val_stats = evaluate_loss(
            model=model, loader=val_loader, device=device,
            theta_mean_t=theta_mean_t, theta_std_t=theta_std_t,
            params=params, lambda_fk=lambda_fk,
        )
        final_val_loss = val_stats["loss"]

        if final_val_loss < best_val_loss - min_delta:
            best_val_loss = final_val_loss
            best_epoch = epoch
            no_improvement_count = 0
        else:
            no_improvement_count += 1

        if no_improvement_count >= patience:
            early_stopped = True
            break

    return {
        "final_train_loss": final_train_loss,
        "final_val_loss": final_val_loss,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "total_epochs_trained": epoch,
        "early_stopped": early_stopped,
    }


lr_sweep_results = []

for lr in learning_rates_to_try:
    print(f"\n=== Learning rate: {lr} ===")

    sweep_model = MDN3R3D(
        input_dim=params.input_dim,
        output_dim=ANGLE_ENCODING_DIM,
        hidden_size=config["hidden_size"],
        n_components=config["n_components"],
    )
    sweep_optimizer = torch.optim.Adam(sweep_model.parameters(), lr=lr)

    sweep_result = train_mdn_in_memory(
        model=sweep_model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=sweep_optimizer,
        device=device,
        params=params,
        theta_mean=processed["theta_mean"],
        theta_std=processed["theta_std"],
        max_epochs=config["max_epochs"],
        patience=config["patience"],
        min_delta=config["min_delta"],
        lambda_fk=config["lambda_fk"],
    )

    sweep_test_metrics, _ = evaluate_best_of_k(
        model=sweep_model, loader=test_loader, device=device, params=params,
        theta_mean=processed["theta_mean"], theta_std=processed["theta_std"],
    )

    lr_sweep_results.append({
        "learning_rate": lr,
        "final_train_loss": sweep_result["final_train_loss"],
        "final_val_loss": sweep_result["final_val_loss"],
        "best_val_loss": sweep_result["best_val_loss"],
        "best_epoch": sweep_result["best_epoch"],
        "total_epochs_trained": sweep_result["total_epochs_trained"],
        "early_stopped": sweep_result["early_stopped"],
        "test_best_of_k_angle_rmse": sweep_test_metrics["best_of_k_angle_rmse"],
        "test_best_of_k_pose_rmse": sweep_test_metrics["best_of_k_pose_rmse"],
    })

    print(
        f"final_train_loss={sweep_result['final_train_loss']:.6f}  "
        f"final_val_loss={sweep_result['final_val_loss']:.6f}  "
        f"best_val_loss={sweep_result['best_val_loss']:.6f} (epoch {sweep_result['best_epoch']})"
    )
    print(
        f"test_angle_rmse={sweep_test_metrics['best_of_k_angle_rmse']:.6f}  "
        f"test_pose_rmse={sweep_test_metrics['best_of_k_pose_rmse']:.6f}"
    )

print("\n=== Learning Rate Sweep Summary ===")
lr_sweep_df = pd.DataFrame(lr_sweep_results)
lr_sweep_df

## 11. Evaluate on Test Set (Best-of-K)

In [ ]:
best_model_path = experiment_dirs["models"] / "best_model.pt"
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)
logger.info("Loaded best model for final evaluation.")

test_metrics, test_predictions = evaluate_best_of_k(
    model=model, loader=test_loader, device=device, params=params,
    theta_mean=processed["theta_mean"], theta_std=processed["theta_std"],
)

final_metrics = {
    **test_metrics,
    "best_val_loss": train_result["best_val_loss"],
    "best_epoch": train_result["best_epoch"],
    "total_epochs_trained": train_result["total_epochs_trained"],
    "early_stopped": train_result["early_stopped"],
    "timestamp": datetime.now().isoformat(),
}

save_final_metrics(final_metrics, experiment_dirs["metrics"])

save_npz(test_predictions, experiment_dirs["predictions"] / "test_predictions.npz")
save_prediction_tables(test_predictions, experiment_dirs["predictions"], config["n_components"], prefix="test")

logger.info(f"Test best-of-K angle MSE: {test_metrics['best_of_k_angle_mse']:.6f}")
logger.info(f"Test best-of-K pose MSE: {test_metrics['best_of_k_pose_mse']:.6f}")

pd.DataFrame([test_metrics])

## 12. Visualizations

In [ ]:
save_training_plots(train_result["history"], experiment_dirs["plots"])

In [ ]:
save_dataset_plots(processed["features_original"], experiment_dirs["plots"])

In [ ]:
save_prediction_plots(test_predictions, experiment_dirs["plots"], config["n_components"], prefix="test")

## 13. Generalization Check: Unseen Data

Generate 1000 freshly sampled (never seen during training) poses and evaluate the trained model on them, using the *same* normalization statistics computed from the training set.

In [ ]:
logger.info("Running unseen-data experiment with 1000 samples.")

unseen_features, unseen_targets = generate_dataset(n_samples=1000, params=params, seed=config["unseen_seed"])

feature_mean = processed["feature_mean"]
feature_std = processed["feature_std"]
theta_mean = processed["theta_mean"]
theta_std = processed["theta_std"]

unseen_targets_sincos = encode_theta_to_sincos(unseen_targets)

unseen_features_norm = (unseen_features - feature_mean) / feature_std
unseen_targets_norm = (unseen_targets_sincos - theta_mean) / theta_std

unseen_dataset = {
    "features_original": unseen_features.astype(np.float32),
    "targets_original": unseen_targets.astype(np.float32),
    "features_normalized": unseen_features_norm.astype(np.float32),
    "targets_normalized": unseen_targets_norm.astype(np.float32),
    "feature_mean": feature_mean, "feature_std": feature_std,
    "theta_mean": theta_mean, "theta_std": theta_std,
}

save_npz(unseen_dataset, experiment_dirs["unseen_dataset"] / "unseen_dataset.npz")

unseen_loader = DataLoader(
    IKDataset3D(unseen_features_norm, unseen_features, unseen_targets_norm),
    batch_size=config["batch_size"], shuffle=False,
)

unseen_metrics, unseen_predictions = evaluate_best_of_k(
    model=model, loader=unseen_loader, device=device, params=params,
    theta_mean=theta_mean, theta_std=theta_std,
)

save_npz(unseen_predictions, experiment_dirs["unseen_predictions"] / "unseen_predictions.npz")
save_json(unseen_metrics, experiment_dirs["unseen_metrics"] / "unseen_metrics.json")
pd.DataFrame([unseen_metrics]).to_csv(experiment_dirs["unseen_metrics"] / "unseen_metrics.csv", index=False)

save_prediction_tables(unseen_predictions, experiment_dirs["unseen_tables"], config["n_components"], prefix="unseen")
save_prediction_plots(unseen_predictions, experiment_dirs["unseen_plots"], config["n_components"], prefix="unseen")

logger.info(f"Unseen best-of-K angle MSE: {unseen_metrics['best_of_k_angle_mse']:.6f}")
logger.info(f"Unseen best-of-K pose MSE: {unseen_metrics['best_of_k_pose_mse']:.6f}")

pd.DataFrame([unseen_metrics])

## 14. Key Diagnostics Summary

Everything you actually need to decide the next step, in one place:

- **Requirement check** — mean angle MAE and best-of-K pose RMSE against the thesis spec (< 5&deg;, < 5&nbsp;mm), for both the test set and the unseen-data generalization check.
- **Bias/variance snapshot** — final-epoch train/val loss gap, so you don't have to re-read the training curve manually.
- **Mixture health** — how evenly the K mixture components are used (a couple of components dominating suggests the mixture isn't representing the solution space well).
- **theta3 error diagnostics** — the programmatic version of the manual check from the teaching workspace's error-analysis lesson: what fraction of errors are a **branch confusion** (wrong elbow-up/down sign) versus correlated with the elbow's **ill-conditioning** (`theta3 -> 0`), plus whether the mixture is confidently wrong or genuinely uncertain on its worst predictions.

Re-run this cell after any training change (sin/cos encoding, LR, capacity, `lambda_fk`, ...) to see whether it actually moved the numbers that matter.

In [ ]:
REQUIRED_MEAN_ANGLE_DEG = 5.0
REQUIRED_POSE_MM = 5.0
METERS_TO_MM = 1000.0  # assumes L1/L2/L3 are in meters — rescale this if your thesis uses different units

RAD2DEG = 180.0 / math.pi

# --- 1. Requirement check: mean angle error < 5 deg, pose distance < 5 mm ---
requirement_rows = []
for split_name, split_metrics in [("test", test_metrics), ("unseen", unseen_metrics)]:
    mean_angle_mae_deg = np.mean([
        split_metrics["theta1_mae"], split_metrics["theta2_mae"], split_metrics["theta3_mae"],
    ]) * RAD2DEG
    pose_rmse_mm = split_metrics["best_of_k_pose_rmse"] * METERS_TO_MM

    requirement_rows.append({
        "split": split_name,
        "theta1_mae_deg": split_metrics["theta1_mae"] * RAD2DEG,
        "theta2_mae_deg": split_metrics["theta2_mae"] * RAD2DEG,
        "theta3_mae_deg": split_metrics["theta3_mae"] * RAD2DEG,
        "mean_angle_mae_deg": mean_angle_mae_deg,
        "best_of_k_angle_rmse_deg": split_metrics["best_of_k_angle_rmse"] * RAD2DEG,
        "best_of_k_pose_rmse_mm": pose_rmse_mm,
        "meets_angle_req_lt_5deg": mean_angle_mae_deg < REQUIRED_MEAN_ANGLE_DEG,
        "meets_pose_req_lt_5mm": pose_rmse_mm < REQUIRED_POSE_MM,
    })

requirement_df = pd.DataFrame(requirement_rows)

# --- 2. Bias/variance snapshot (final epoch train vs val loss gap) ---
history_df = pd.DataFrame(train_result["history"])
final_row = history_df.iloc[-1]
bias_variance_gap = final_row["val_loss"] - final_row["train_loss"]

# --- 3. Mixture health: is component usage spread out, or collapsed onto a couple of components? ---
usage_df = component_usage_dataframe(test_predictions, config["n_components"])

# --- 4. theta3 diagnostics: branch confusion (wrong elbow-up/down sign) vs ill-conditioning ---
theta3_true = test_predictions["y_true_real"][:, 2]
theta3_pred = test_predictions["predicted_best_fk_real"][:, 2]
theta3_abs_error = np.abs(theta3_pred - theta3_true)

sign_flipped = np.sign(theta3_pred) != np.sign(theta3_true)
branch_confusion_rate = float(np.mean(sign_flipped))

singularity_proxy = np.abs(np.sin(theta3_true))  # -> 0 near full extension/fold (elbow singularity)
correct_branch_mask = ~sign_flipped
ill_conditioning_corr = float(
    np.corrcoef(theta3_abs_error[correct_branch_mask], singularity_proxy[correct_branch_mask])[0, 1]
) if correct_branch_mask.sum() > 1 else float("nan")

mixture_confidence = np.max(test_predictions["mixture_probabilities"], axis=1)
worst_mask = theta3_abs_error >= np.quantile(theta3_abs_error, 0.9)
confidence_on_worst_decile = float(np.mean(mixture_confidence[worst_mask]))
confidence_overall = float(np.mean(mixture_confidence))

# --- Print everything ---
print("=== Requirement check (mean angle MAE < 5 deg, best-of-K pose RMSE < 5 mm) ===")
print(requirement_df.to_string(index=False))

print(f"\n=== Bias/variance snapshot (epoch {int(final_row['epoch'])}) ===")
print(f"train_loss={final_row['train_loss']:.4f}  val_loss={final_row['val_loss']:.4f}  gap={bias_variance_gap:.4f}")
print(f"best_epoch={train_result['best_epoch']}  total_epochs_trained={train_result['total_epochs_trained']}  early_stopped={train_result['early_stopped']}")

print("\n=== Mixture component usage (test set) ===")
print(usage_df.to_string(index=False))

print("\n=== theta3 error diagnostics (test set) ===")
print(f"Branch confusion rate (wrong elbow-up/down sign): {branch_confusion_rate:.1%}")
print(f"Ill-conditioning correlation (|abs error| vs |sin(theta3_true)|, correct-branch samples only): {ill_conditioning_corr:.3f}")
print(f"  -> more negative = stronger evidence error grows near theta3 -> 0 (full extension/fold singularity)")
print(f"Mixture confidence, worst 10% of theta3 errors: {confidence_on_worst_decile:.3f}  (overall: {confidence_overall:.3f})")
print(f"  -> confidence similar to overall = model is 'confidently wrong' near failures, not uncertain")

print("\n=== What to try next, based on the above ===")
if branch_confusion_rate > 0.05:
    print(f"- Branch confusion affects {branch_confusion_rate:.1%} of samples: consider more mixture components,")
    print("  a diversity-encouraging loss term, or a multi-head architecture with one head per elbow branch.")
if not np.isnan(ill_conditioning_corr) and ill_conditioning_corr < -0.2:
    print("- Error correlates with proximity to the elbow singularity: consider oversampling near theta3=0")
    print("  during dataset generation, or down-weighting the FK loss there (its gradient is also ill-conditioned).")
if branch_confusion_rate <= 0.05 and (np.isnan(ill_conditioning_corr) or ill_conditioning_corr >= -0.2):
    print("- Neither failure mode dominates clearly — re-check the two numbers above after re-running with sin/cos")
    print("  encoding (this cell), since it should reduce both relative to the earlier raw-radian version.")

requirement_df

## 15. Save & Download Results

Zips the whole `outputs/experiments/<timestamp>/` folder (models, metrics, plots, predictions) and triggers a download. If you'd rather keep results in Drive, mount it first (`from google.colab import drive; drive.mount('/content/drive')`) and copy `experiment_dirs["root"]` there instead.

In [ ]:
logger.info("Pipeline complete.")

zip_path = shutil.make_archive(str(experiment_dirs["root"]), "zip", root_dir=experiment_dirs["root"])
print(f"Zipped results to: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab — find the zip locally at the path above.")